In [ ]:
import pandas as pd
import requests
from datetime import datetime, timedelta
import os
from pathlib import Path

# Create data directories if they don't exist
data_dir = Path('../data')
raw_dir = data_dir / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

# Weather API configuration
API_KEY = os.getenv('WEATHER_API_KEY')
BASE_URL = 'https://api.weatherapi.com/v1/history.json'

def get_major_cities():
    """
    Returns a list of major cities around the world.
    """
    cities = [
        "New York", "London", "Paris", "Tokyo", "Beijing", "Dubai",
        "Singapore", "Hong Kong", "Mumbai", "São Paulo", "Mexico City",
        "Cairo", "Moscow", "Berlin", "Sydney", "Toronto", "Istanbul",
        "Seoul", "Bangkok", "Jakarta", "Manila", "Lagos", "Delhi",
        "Shanghai", "Rio de Janeiro", "Buenos Aires", "Los Angeles",
        "Chicago", "Madrid", "Rome", "Amsterdam", "Vienna", "Stockholm",
        "Copenhagen", "Oslo", "Helsinki", "Warsaw", "Prague", "Budapest",
        "Athens", "Lisbon", "Brussels", "Dublin", "Edinburgh", "Vancouver",
        "Montreal", "Cape Town", "Johannesburg", "Nairobi", "Tel Aviv"
    ]
    return cities

def fetch_weather_data(start_date, end_date, location):
    """
    Fetch historical weather data for a given location and date range.
    
    Args:
        start_date (datetime): Start date for data collection
        end_date (datetime): End date for data collection
        location (str): Location name or coordinates
        
    Returns:
        pd.DataFrame: Weather data
    """
    data = []
    current_date = start_date
    
    while current_date <= end_date:
        params = {
            'key': API_KEY,
            'q': location,
            'dt': current_date.strftime('%Y-%m-%d')
        }
        
        try:
            response = requests.get(BASE_URL, params=params)
            response.raise_for_status()
            daily_data = response.json()
            data.append(daily_data)
            
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data for {current_date}: {e}")
            
        current_date += timedelta(days=1)
    
    return pd.DataFrame(data)

# Example usage
start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 12, 31)

# Get list of major cities
cities = get_major_cities()

# Fetch data for each city
for city in cities:
    print(f"Fetching data for {city}...")
    weather_df = fetch_weather_data(start_date, end_date, city)
    
    # Save raw data
    raw_file_path = raw_dir / f'weather_data_{city.replace(" ", "_")}_{start_date.strftime("%Y%m%d")}_{end_date.strftime("%Y%m%d")}.parquet'
    weather_df.to_parquet(raw_file_path)
    print(f"Data saved for {city} at: {raw_file_path}")

print("Data collection complete for all cities.")
